In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
import string
import time
import matplotlib.pyplot as plt

D:\Anaconda\envs\stock1\lib\site-packages\torch\cuda\__init__.py:83: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at  ..\c10\cuda\CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [8]:
all_letters = string.ascii_letters + ".,;"  #覆盖大小写字母和常见标点，每个字符成为一个特征维度
n_letters = len(all_letters)
print(all_letters,n_letters)
categorys = ['Italian','English','Arabic','Spanish','Scottish','Irish','Chinese','Vietnamese','Frence','Greek','Dutch','Korean','Polish','Portuguese','Russian','Czech','German']
categorynum = len(categorys)
print(categorys)
print(categorynum)


abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ.,; 55
['Italian', 'English', 'Arabic', 'Spanish', 'Scottish', 'Irish', 'Chinese', 'Vietnamese', 'Frence', 'Greek', 'Dutch', 'Korean', 'Polish', 'Portuguese', 'Russian', 'Czech', 'German']
17


In [3]:
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [4]:
#读取元数据到内存
def read_data(file_path):
    ##读取
    my_list_x,my_list_y = [],[]
    with open(file_path,'r',encoding = 'utf-8') as f:
        for line in f.readlines():
            if len(line) <= 5:
                continue
            #走到这，说明改行数据有效
            x,y = line.strip().split('\t')    #过滤空行、无效行
            my_list_x.append(x)
            my_list_y.append(y)
    print(f'my_list_x:{len(my_list_x)}')
    print(f'my_list_y:{len(my_list_y)}')
    return my_list_x,my_list_y

In [6]:
read_data('name_classfication.txt')  #返回两个平行列表，人名列表，标签列表。

my_list_x:20074
my_list_y:20074


(['Abl',
  'Adsit',
  'Ajdrna',
  'Alt',
  'Antonowitsch',
  'Antonowitz',
  'Bacon',
  'Ballalatak',
  'Ballaltick',
  'Bartonova',
  'Bastl',
  'Baroch',
  'Benesch',
  'Betlach',
  'Biganska',
  'Bilek',
  'Blahut',
  'Blazek',
  'Blazek',
  'Blazejovsky',
  'Blecha',
  'Bleskan',
  'Blober',
  'Bock',
  'Bohac',
  'Bohunovsky',
  'Bolcar',
  'Borovka',
  'Borovski',
  'Borowski',
  'Borovsky',
  'Brabbery',
  'Brezovjak',
  'Brousil',
  'Bruckner',
  'Buchta',
  'Cablikova',
  'Camfrlova',
  'Cap',
  'Cerda',
  'Cermak',
  'Chermak',
  'Cermak',
  'Cernochova',
  'Cernohous',
  'Cerny',
  'Cerney',
  'Cerny',
  'Cerv',
  'Cervenka',
  'Chalupka',
  'Charlott',
  'Chemlik',
  'Chicken',
  'Chilar',
  'Chromy',
  'Cihak',
  'Clineburg',
  'Klineberg',
  'Cober',
  'Colling',
  'Cvacek',
  'Czabal',
  'Damell',
  'Demall',
  'Dehmel',
  'Dana',
  'Dejmal',
  'Dempko',
  'Demko',
  'Dinko',
  'Divoky',
  'Dolejsi',
  'Dolezal',
  'Doljs',
  'Dopita',
  'Drassal',
  'Driml',
  'Duyava',

In [17]:
#原始数据 → 数据集对象（tensordataset） → 数据迭代器(dataloader)
class NameClassDataset(Dataset):
    def __init__(self,my_list_x,my_list_y):
        self.my_list_x = my_list_x   #存储样本数据列
        self.my_list_y = my_list_y   #存储标签数据列
        self.sample_len = len(my_list_x) #计算样本总数并存储，20074
    # 定义函数，用于获取样本总数，外界用len(NameClassDataset)时，自动触发（魔法方法）
    def __len__(self):
        return self.sample_len

    #定义函数，实现根据指定索引，获取其对应的样本
    def __getitem__(self,index):
        #1 索引 边界校验，确保索引在合法范围
        index = min(max(index,0),self.sample_len - 1)

        #按照索引获取原始样本和标签
        x = self.my_list_x[index]
        y = self.my_list_y[index]

        #人名数据转化为one-hot编码
        #生成全零向量
        tensor_x = torch.zeros(len(x),n_letters)
        for li,letter in enumerate(x):
            #获取字母在全局字母表中的所有位置，例如字母'D‘
            letter_index = all_letters.find(letter)
            if letter_index != -1:  # 防止字母不在表中
                tensor_x[li][letter_index] = 1
            tensor_x[li][letter_index] = 1
        tensor_y = torch.tensor(categorys.index(y))
        return tensor_x, tensor_y


In [18]:
# 定义函数，获取数据加载器对象
def get_dataloader():
    my_list_x,my_list_y = read_data('name_classfication.txt')
    name_class_dataset = NameClassDataset(my_list_x,my_list_y)
    #创建数据加载器对象，用于批量加载和处理数据
    my_dataloader = DataLoader(name_class_dataset,batch_size = 1,shuffle = False)
    #测试数据加载器，打印第一批数据的形状和内容
    for x,y in my_dataloader:
        print(f'x.shape:{x.shape},x:{x}')
        print(f'y.shape:{y.shape},y:{y}')
        break
    #优化1：可以把上述的数据加载器返回，后续之间调用
    return my_dataloader

In [19]:
get_dataloader()

my_list_x:20074
my_list_y:20074
x.shape:torch.Size([1, 3, 55]),x:tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.]]])
y.shape:torch.Size([1]),y:tensor([15])


In [1]:
import torch
import torch.nn as nn

D:\Anaconda\envs\stock1\lib\site-packages\torch\cuda\__init__.py:83: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at  ..\c10\cuda\CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
#创建数据 → 模拟模型输出的原始分时（logits），表示3个分类的预测值，即：全连接层的处理后的结果
output = torch.tensor([3.2,5.1,-1.7])

In [3]:
#思路→ logsoftmax()函数计算
log_softmax = nn.LogSoftmax(dim=0)
#具体的计算：先softmax(),然后log()
log_probs = log_softmax(output)
#打印结果
print(f'计算结果（对数概率）:{log_probs}')

计算结果（对数概率）:tensor([-2.0404, -0.1404, -6.9404])


In [4]:
#思路2：手动验算，先softmax()，然后log（）
softmax = torch.softmax(output,dim=0)
print(softmax)

log_softmax_probs = torch.log(softmax)
print(f'计算结果:{log_softmax_probs}')  

tensor([0.1300, 0.8690, 0.0010])
计算结果:tensor([-2.0404, -0.1404, -6.9404])


In [5]:
#定义函数，获取数据加载器对象 → Tensor --- tensordataset---dataloader
class My_RNN(nn.Module):
    def __init__(self,input_size,hidden_size,output_size,n_layers = 1):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers

        #定义RNN层，接收输入特征和输出隐藏状态
        self.rnn = nn.RNN(self.input_size,self.hidden_size,self.n_layers)   # 57,128,1
        #定义全连接层，将RNN的隐藏状态转换为输出
        self.linear = nn.Linear(self.hidden_size,self.output_size)
        #定义激活函数，将输入类别 → 概率分布
        self.softmax = nn.LogSoftmax(dim = -1)

    #前向传播函数
    #参1： input输入张量  形状为[seq_len,batch_size,input_size]
    #参2：hidden隐藏状态，[n_layers,batch_size,hidden_size]
    def forward(self,input,hidden):
        #调整输入张量，添加：batch_size
        input = input.unsqueeze(1)

        #通过RNN计算
        #output：所有时间步的隐藏状态，hidden：最后1个时间步的隐藏状态
        output,hn = self.rnn(input,hidden)
        #2.3 提取最后一个时间步的隐藏状态
        tmp_output = output[-1]
        # 通过全连接层，获取输出
        tmp_output = self.linear(tmp_output)
        #数据通过激活函数，映射到概率分布，并返回
        return self.softmax(tmp_output),hn
    #初始化隐藏层状态，创建全0初始化隐藏状态

    def init_hidden(self):
        return torch.zeros(self.n_layers,1,self.hidden_size)

In [6]:
#RNN模型测试，
def dm_test_myrnn():
    #1、实例化RNN对象
    my_rnn = My_RNN(input_size = 57,hidden_size = 128,output_size = 17)
    #print(f'my_rnn:{my_rnn}')

    #准备测试数据，创建1个随机张量，模拟输入,[seq_len,input_size]
    input = torch.randn(6,57)
    print(f'input:{input.shape}')

    #创建初始隐藏状态
    h0 = my_rnn.init_hidden()

    #测试一次性输入完整的一个样本
    output,hn = my_rnn(input,h0)
    #打印结果
    print(f'输出的形状:{output.shape},输出的内容:{output}')  #[1,17]
    print(f'隐藏状态的形状:{hn.shape},隐藏状态的内容:{hn}')

In [7]:
#搭建LSTM网络
class My_LSTM(nn.Module):
    def __init__(self,input_size,hidden_size,output_size,n_layers = 1):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers

        self.LSTM = nn.LSTM(self.input_size,self.hidden_size,self.n_layers)

        self.linear = nn.Linear(self.hidden_size,self.output_size)
         #定义激活函数，将输入类别 → 概率分布
        self.softmax = nn.LogSoftmax(dim = -1)
    def forward(self,input,hidden,c):
            #调整输入张量，添加：batch_size
        input = input.unsqueeze(1)

        #通过LSTM计算
        #output：所有时间步的隐藏状态，hidden：最后1个时间步的隐藏状态
        output,(hn,cn) = self.LSTM(input,(hidden,c))
        #2.3 提取最后一个时间步的隐藏状态
        tmp_output = output[-1]
        # 通过全连接层，获取输出
        tmp_output = self.linear(tmp_output)
        #数据通过激活函数，映射到概率分布，并返回
        #返回值1：预测的类别概率分布，形状:{batch_size,output_size}
        #返回值2：（最后一个时间步的）隐藏状态张量，形状：[n_layers,batch_size,hidden_size]
        #返回值3：（最后一个时间步的）细胞状态张量，形状：{n_layers,batch_size,hidden_size}
        return self.softmax(tmp_output),hn,cn
    #初始化隐藏层状态，创建全0初始化隐藏状态
    def init_hidden(self):
        hidden = c = torch.zeros(self.n_layers,1,self.hidden_size)
        return hidden,c


In [ ]:
#测试RNN,LSTM,GRU网络模型
def dm_test_rnn_lstm_gru():
    #1、 定义遍历，记录： 输入维度（词向量维度57），隐藏层维度128，输出维度（17，国家数量）
    input_size,n_hidden,output_size = n_letters,128,category_num

    my_list_x,my_list_y = read_data('classification.txt')
    name_class_dataset = NameClassDataset(my_list_x,my_list_y)
    my_dataloader = Dataloader(name_class_dataset,batch_size = 1,shuffle= True)
    my_rnn = My_RNN(input_size,n_hidden,output_size)
    my_lstm = My_LSTM(input_size,n_hidden,output_size)

    #测试RNN模型
    for i,(x,y) in enumerate(my_dataloader):
        print(f'i:{i}')
        print(f'x:{x},x.shape:{x.shape}')
        print(f'y:{y},y.shape:{y.shape}')
        hidden = my_rnn.init_hidden()

        output,hidden = my_rnn(x[0],hidden)
        print(f'RNN输出形状:{output.shape},预测结果:{output}')
        if i == 0:
            break

In [ ]:
def train_rnn():
    my_list_x,my_list_y = read_data('name_classification.txt')
    name_class_dataset = NameClassDataset(my_list_x,my_list_y)
    input_size,n_hidden,output_size = n_letters,128,category_num

    my_rnn = My_RNN(input_size,n_hidden,output_size)
    criterion = nn.NLLLoss()
    optimizer = torch.optim.Adam(my_rnn.parameters(),lr = my_lr)
    #调整过程→ 参数初始化
    start_time = time.time()  # 模型开始训练时间
    total_iter_num = 0   # 已训练的样本数
    total_loss = 0.0 #已训练的损失和
    total_loss_list = [] #每100个样本求一次平均损失
    total_acc_num = 0  #已训练的成本，预测准确总数
    total_acc_list = []  #每100个汤包求一次平均准确率，形成：准确率列表

    for epoch in range(epochs):
        print(f'\n开始{epoch+1}/{epochs}轮训练')
        # 创建数据集加载器对象，随机打乱数据集
        train_dataloader = DataLoader(name_class_dataset,batch_size= 1.shuffle = True)
        for i,(x,y) in enumerate(train_dataloader)):
            output,hidden = my_rnn(x[0],my_rnn.init_hidden())
            my_loss = criterion(output,y)
            #梯度清零，反向传播，优化器
            optimizer.zero_grad()
            my_loss.backward()
            optimizer.step()

            #统计训练结果
            total_iter_num += 1  #训练的样本数 + 1
            total_loss += my_loss.item()  #累计损失之
            total_acc_num += (1 if pred_tag == y else 0)

            #统计：每100个样本求一次平均损失，准确率，形成：损失列表，准确率列表
            if total_iter_num % 100 == 0:
                avg_loss = total_loss /total_iter_num   # 总损失/总样本数

                total_loss_list.append(avg_loss)

            if total_iter_num % 2000 == 0:
                avg_loss = total_loss/total_iter_num
                end_time = int(time.time() - start_time)
                print(f'轮次:{epoch + 1},训练的样本数:{total_iter_num},平均损失:{avg_loss:%.4f}')
    return total_loss,total_time,total_acc,list

            
                

    